In [3]:
import pandas as pd
import os
from osgeo import gdal, ogr, osr

# Load the table
data = pd.read_csv("split_64_perblock.csv")

def get_centroid(tif_path):
    """Extracts the centroid of a georeferenced TIFF file using GDAL."""
    dataset = gdal.Open(tif_path)
    if dataset is None:
        raise FileNotFoundError(f"File not found: {tif_path}")
    
    geo_transform = dataset.GetGeoTransform()
    width = dataset.RasterXSize
    height = dataset.RasterYSize
    
    centroid_x = geo_transform[0] + (width / 2) * geo_transform[1]
    centroid_y = geo_transform[3] + (height / 2) * geo_transform[5]
    
    return centroid_x, centroid_y

# Create a GeoPackage
driver = ogr.GetDriverByName("GPKG")
dataset = driver.CreateDataSource("output.gpkg")
spatial_ref = osr.SpatialReference()
spatial_ref.ImportFromEPSG(3857)  # Set projection to EPSG:3857
layer = dataset.CreateLayer("centroids", geom_type=ogr.wkbPoint, srs=spatial_ref)

# Define fields
field_name = ogr.FieldDefn("image_name", ogr.OFTString)
field_name.SetWidth(50)
layer.CreateField(field_name)

field_split = ogr.FieldDefn("split_name", ogr.OFTString)
field_split.SetWidth(10)
layer.CreateField(field_split)

# Process each row
for _, row in data.iterrows():
    tif_path = os.path.join("dataset_cropped_64_coniferous", f"{row['image_name']}_S2.tif")
    try:
        centroid_x, centroid_y = get_centroid(tif_path)
        
        # Create feature
        feature = ogr.Feature(layer.GetLayerDefn())
        feature.SetField("image_name", row["image_name"])
        feature.SetField("split_name", row["split_name"])
        
        # Set geometry
        point = ogr.Geometry(ogr.wkbPoint)
        point.AddPoint(centroid_x, centroid_y)
        feature.SetGeometry(point)
        layer.CreateFeature(feature)
        feature = None  # Free memory
    except Exception as e:
        print(f"Skipping {tif_path}: {e}")

dataset = None  # Save and close the GeoPackage
print("GeoPackage created successfully.")

GeoPackage created successfully.
